# MCP Server Knowledge Source Quickstart

This notebook walks through the first validation path for the accelerator:

1. Create an MCP Server Knowledge Source.
2. Attach it to a Knowledge Base.
3. Run retrieve with `knowledgeSourceParams`.
4. Inspect `activity`, `references`, and `sourceData`.

The default MCP endpoint is the public Microsoft Learn MCP server: `https://learn.microsoft.com/api/mcp`.

Outputs are intentionally empty in the committed notebook. Clear outputs before committing live test runs.

## Runtime Safety

The notebook defaults to dry-run mode and prints payloads only. To call Azure AI Search, set `RUN_LIVE_CALLS=true` in an untracked `.env` file or in your notebook environment.

For private local testing with another untracked env file, set `LIVE_KS_ENV_FILE=/path/to/.env`. Do not commit tenant values, API keys, tokens, or live retrieve outputs.

In [ ]:
from __future__ import annotations

import json
import os
import sys
import urllib.error
import urllib.request
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path.cwd().parent

sys.path.insert(0, str(REPO_ROOT / "src"))

from ks_factory import create_knowledge_base, create_mcp_server_knowledge_source

def load_env_file(path: Path) -> dict[str, str]:
    values: dict[str, str] = {}
    if not path.exists():
        return values
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values

env_file = Path(os.getenv("LIVE_KS_ENV_FILE", ".env")).expanduser()
file_env = load_env_file(env_file)

def config(*names: str, default: str | None = None) -> str | None:
    for name in names:
        value = os.getenv(name) or file_env.get(name)
        if value:
            return value
    return default

SEARCH_ENDPOINT = config("SEARCH_ENDPOINT", "AZURE_SEARCH_ENDPOINT")
SEARCH_API_KEY = config("SEARCH_API_KEY", "AZURE_SEARCH_API_KEY")
API_VERSION = config("SEARCH_API_VERSION", "AZURE_SEARCH_API_VERSION", default="2026-05-01-preview")
KNOWLEDGE_BASE_NAME = config("KNOWLEDGE_BASE_NAME", "KB_NAME", default="live-knowledge-sources-kb")
MCP_KS_NAME = config("MCP_KNOWLEDGE_SOURCE_NAME", "DEFAULT_KS_NAME", default="microsoft-learn-mcp-ks")
MCP_SERVER_URL = config("MCP_SERVER_URL", "MCP_URL", default="https://learn.microsoft.com/api/mcp")
MCP_TOOL_NAME = config("MCP_TOOL_NAME", default="microsoft_docs_search")
RUN_LIVE_CALLS = (config("RUN_LIVE_CALLS", default="false") or "false").lower() == "true"

DEFAULT_MCP_SERVER_URL = "https://learn.microsoft.com/api/mcp"

print(json.dumps({
    "envFileConfigured": env_file.exists(),
    "hasSearchEndpoint": bool(SEARCH_ENDPOINT),
    "hasSearchApiKey": bool(SEARCH_API_KEY),
    "apiVersion": API_VERSION,
    "knowledgeBaseName": KNOWLEDGE_BASE_NAME,
    "mcpKnowledgeSourceName": MCP_KS_NAME,
    "usesDefaultMicrosoftLearnMcp": MCP_SERVER_URL == DEFAULT_MCP_SERVER_URL,
    "hasMcpServerUrl": bool(MCP_SERVER_URL),
    "mcpToolName": MCP_TOOL_NAME,
    "runLiveCalls": RUN_LIVE_CALLS
}, indent=2))

## Build The Payloads

These payloads match the REST files under `samples/rest/`.

In [ ]:
mcp_knowledge_source = create_mcp_server_knowledge_source(
    name=MCP_KS_NAME,
    server_url=MCP_SERVER_URL,
    tool_name=MCP_TOOL_NAME,
    description="Microsoft Learn MCP grounding source for official documentation.",
)

mcp_only_knowledge_base = create_knowledge_base(
    name=KNOWLEDGE_BASE_NAME,
    knowledge_source_names=[MCP_KS_NAME],
    description="Knowledge Base for validating MCP Server live grounding before adding tenant-specific sources.",
    retrieval_instructions=(
        "Use the Microsoft Learn MCP Server knowledge source for questions about "
        "Azure AI Search, Foundry IQ, and Microsoft documentation."
    ),
)

def retrieve_payload(question: str) -> dict:
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": question,
                    }
                ],
            }
        ],
        "includeActivity": True,
        "knowledgeSourceParams": [
            {
                "kind": "mcpServer",
                "knowledgeSourceName": MCP_KS_NAME,
                "includeReferences": True,
                "includeReferenceSourceData": True,
            }
        ],
        "outputMode": "answerSynthesis",
        "retrievalReasoningEffort": {
            "kind": "low",
        },
        "maxRuntimeInSeconds": 60,
    }

def scrub_for_display(payload: dict) -> dict:
    safe = json.loads(json.dumps(payload))
    parameters = safe.get("mcpServerParameters")
    if parameters and parameters.get("serverURL") != DEFAULT_MCP_SERVER_URL:
        parameters["serverURL"] = "<configured-mcp-server-url>"
    return safe

print(json.dumps({
    "knowledgeSource": scrub_for_display(mcp_knowledge_source),
    "knowledgeBase": mcp_only_knowledge_base,
    "retrieve": retrieve_payload("What must be configured to create an Azure AI Search MCP Server knowledge source?")
}, indent=2))

## Optional Live Calls

The cell below only runs when `RUN_LIVE_CALLS=true`. It creates or updates the MCP Knowledge Source and Knowledge Base, then runs a retrieve request.

In [ ]:
def search_url(path: str) -> str:
    if not SEARCH_ENDPOINT:
        raise ValueError("SEARCH_ENDPOINT or AZURE_SEARCH_ENDPOINT is required for live calls.")
    return f"{SEARCH_ENDPOINT.rstrip('/')}/{path.lstrip('/')}?api-version={API_VERSION}"

def send_json(method: str, url: str, payload=None) -> dict:
    if not SEARCH_API_KEY:
        raise ValueError("SEARCH_API_KEY or AZURE_SEARCH_API_KEY is required for this API-key quickstart.")
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        url,
        data=body,
        method=method,
        headers={
            "api-key": SEARCH_API_KEY,
            "Content-Type": "application/json",
            "Prefer": "return=representation",
        },
    )
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            text = response.read().decode("utf-8")
            return json.loads(text) if text else {}
    except urllib.error.HTTPError as exc:
        error_text = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"{method} {url} failed with HTTP {exc.code}: {error_text}") from exc

def summarize_reference(reference: dict) -> dict:
    return {
        "type": reference.get("type"),
        "title": reference.get("title"),
        "toolName": reference.get("toolName"),
        "hasSourceData": "sourceData" in reference,
    }

if not RUN_LIVE_CALLS:
    print("Dry run only. Set RUN_LIVE_CALLS=true to call Azure AI Search.")
else:
    ks_result = send_json("PUT", search_url(f"knowledgesources/{MCP_KS_NAME}"), mcp_knowledge_source)
    kb_result = send_json("PUT", search_url(f"knowledgebases/{KNOWLEDGE_BASE_NAME}"), mcp_only_knowledge_base)
    retrieve_result = send_json(
        "POST",
        search_url(f"knowledgebases/{KNOWLEDGE_BASE_NAME}/retrieve"),
        retrieve_payload("What must be configured to create an Azure AI Search MCP Server knowledge source?"),
    )
    print(json.dumps({
        "knowledgeSourceName": ks_result.get("name"),
        "knowledgeBaseName": kb_result.get("name"),
        "activity": retrieve_result.get("activity", []),
        "references": [summarize_reference(ref) for ref in retrieve_result.get("references", [])],
    }, indent=2))

## Expected Trace

A successful MCP test should show an `mcpServer` activity record and MCP references. For more query ideas, see `docs/08-test-queries.md`.

In [ ]:
expected_trace = {
    "activity": {
        "type": "mcpServer",
        "knowledgeSourceName": MCP_KS_NAME,
        "mcpServerArguments": {
            "toolName": MCP_TOOL_NAME,
        },
    },
    "reference": {
        "type": "mcpServer",
        "sourceData": "present when includeReferenceSourceData is true",
    },
}

print(json.dumps(expected_trace, indent=2))